# Length-normalized mutation frequencies
### Pick **any** cBioPortal study and compare raw vs CDS-length-normalized mutation frequency

**MSK CBSP Hackathon 2026 — team veggenes**

Longer genes accumulate more mutations simply because they present a bigger target, so a raw "% of samples mutated" ranking is biased toward big genes. This notebook:

1. Lets you **choose a study** (search by keyword, or use the dropdown).
2. **Extracts** its mutations and computes per-gene frequency.
3. **Fetches gene lengths** (canonical **CDS** length) from **Ensembl**.
4. **Normalizes** frequency by CDS length and shows raw vs normalized rankings.

Everything below is wrapped in functions, so switching cohorts is a one-line change. APIs: [cBioPortal](https://www.cbioportal.org/api) (`bravado` + `requests`) and [Ensembl REST](https://rest.ensembl.org).

In [ ]:
# First run only (uncomment):
# !pip install bravado pandas matplotlib seaborn requests ipywidgets

from bravado.client import SwaggerClient
from IPython.display import display
import requests, time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("white")
sns.set_context("notebook")

# ---- Defaults (override per run) -----------------------------------
STUDY_ID = "luad_mskimpact_2021"   # default study; change it or use the picker below
TOP_N    = 100                      # keep the top-N most frequently mutated genes
CBIO     = "https://www.cbioportal.org/api"
ENSEMBL  = "https://rest.ensembl.org"
# --------------------------------------------------------------------

cbioportal = SwaggerClient.from_url(
    f"{CBIO}/v2/api-docs",
    config={"validate_requests": False,
            "validate_responses": False,
            "validate_swagger_spec": False})
print(cbioportal)

## 1. Choose a study

`find_studies(keyword)` searches all cBioPortal studies (by name or id) and returns the matches that have mutation data, sorted by size. Use it to discover the `studyId` you want.

In [ ]:
def find_studies(keyword="", with_mutations=True):
    """Return a DataFrame of studies whose name or id matches `keyword`."""
    studies = requests.get(f"{CBIO}/studies", params={"projection": "SUMMARY"},
                           headers={"Accept": "application/json"}, timeout=60).json()
    df = pd.DataFrame(studies)
    if keyword:
        k = keyword.lower()
        mask = (df["name"].str.lower().str.contains(k)
                | df["studyId"].str.lower().str.contains(k))
        df = df[mask]
    cols = ["studyId", "name", "allSampleCount", "cancerTypeId"]
    df = df[[c for c in cols if c in df.columns]].copy()
    return df.sort_values("allSampleCount", ascending=False).reset_index(drop=True)

# Example: all lung studies
find_studies("lung").head(20)

### Optional: pick the study from a dropdown

Set `SEARCH` to any keyword (e.g. `"lung"`, `"melanoma"`, `"msk"`, `"tcga"`). Run the cell, choose from the dropdown, and the selection is stored in `STUDY_ID`. If `ipywidgets` isn't available, just set `STUDY_ID` by hand.

In [ ]:
SEARCH = "lung"   # <- keyword to populate the dropdown

try:
    import ipywidgets as widgets
    _opts_df = find_studies(SEARCH)
    _options = [(f"{r['name']}  ({r.studyId}, n={r.allSampleCount})", r.studyId)
                for _, r in _opts_df.iterrows()]
    _dd = widgets.Dropdown(options=_options, description="Study:",
                           layout=widgets.Layout(width="90%"))
    def _on_change(change):
        global STUDY_ID
        if change["name"] == "value":
            STUDY_ID = change["new"]
            print("STUDY_ID =", STUDY_ID)
    _dd.observe(_on_change)
    STUDY_ID = _dd.value          # initialise to the first option
    display(_dd)
    print("STUDY_ID =", STUDY_ID, "  (change the dropdown to update)")
except ImportError:
    print("ipywidgets not installed -- set STUDY_ID manually, e.g. STUDY_ID = 'luad_tcga_pub'")

## 2. Mutation frequency per gene (for any study)

`get_mutation_frequencies` is study-agnostic: it discovers the study's mutation profile and the correct denominator (the *sequenced* sample list — samples profiled for mutations, including those with zero mutations), then returns a per-gene table.

In [ ]:
def _mutation_profile_id(study_id):
    profs = requests.get(f"{CBIO}/studies/{study_id}/molecular-profiles",
                         headers={"Accept": "application/json"}, timeout=60).json()
    for p in profs:
        if p.get("molecularAlterationType") == "MUTATION_EXTENDED":
            return p["molecularProfileId"]
    raise ValueError(f"No mutation profile found for study '{study_id}'.")

def _profiled_sample_list(study_id):
    """Prefer the '_sequenced' list (mutation-profiled samples); fall back to '_all'."""
    lists = requests.get(f"{CBIO}/studies/{study_id}/sample-lists",
                         headers={"Accept": "application/json"}, timeout=60).json()
    ids = {l["sampleListId"] for l in lists}
    for suffix in ("_sequenced", "_all"):
        if f"{study_id}{suffix}" in ids:
            return f"{study_id}{suffix}"
    return lists[0]["sampleListId"]

def get_mutation_frequencies(study_id, top_n=None):
    profile_id = _mutation_profile_id(study_id)
    sample_list_id = _profiled_sample_list(study_id)

    # denominator: number of profiled samples
    sl = requests.get(f"{CBIO}/sample-lists/{sample_list_id}",
                      params={"projection": "DETAILED"},
                      headers={"Accept": "application/json"}, timeout=60).json()
    n_profiled = len(sl["sampleIds"])

    # all mutations (DETAILED -> gene field populated)
    mutations = cbioportal.Mutations.getMutationsInMolecularProfileBySampleListIdUsingGET(
        molecularProfileId=profile_id,
        sampleListId=sample_list_id,
        projection="DETAILED",
    ).result()

    mdf = pd.DataFrame.from_dict([
        dict({k: getattr(m, k) for k in dir(m)},
             **{k: getattr(m.gene, k) for k in dir(m.gene)})
        for m in mutations
    ])

    freq = pd.DataFrame({
        "altered_samples": mdf.groupby("hugoGeneSymbol")["uniqueSampleKey"].nunique(),
        "total_mutations": mdf.groupby("hugoGeneSymbol").size(),
    }).sort_values("altered_samples", ascending=False)
    freq["freq_pct"] = 100 * freq["altered_samples"] / n_profiled

    freq.attrs["study_id"] = study_id
    freq.attrs["n_profiled"] = n_profiled
    print(f"{study_id}: {len(mutations)} mutations across {n_profiled} profiled samples; "
          f"{len(freq)} mutated genes.")
    return freq.head(top_n) if top_n else freq

## 3. Canonical CDS length from Ensembl (for any gene set)

In [ ]:
def _ensembl_post(path, payload, params=None):
    r = requests.post(f"{ENSEMBL}{path}",
                      headers={"Content-Type": "application/json", "Accept": "application/json"},
                      params=params or {}, json=payload, timeout=60)
    r.raise_for_status()
    return r.json()

def _chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

def get_cds_lengths(genes, seq_type="cds"):
    """Map Hugo symbols -> canonical-transcript CDS length (bp). seq_type='protein' for aa."""
    # symbol -> canonical transcript id (version stripped)
    canonical = {}
    for batch in _chunks(list(genes), 50):
        res = _ensembl_post("/lookup/symbol/homo_sapiens", {"symbols": batch})
        for sym, info in res.items():
            ct = info.get("canonical_transcript")
            if ct:
                canonical[sym] = ct.split(".")[0]
        time.sleep(0.2)

    # transcript id -> sequence length
    tx_to_genes = {}
    for sym, tx in canonical.items():
        tx_to_genes.setdefault(tx, []).append(sym)
    lengths = {}
    for batch in _chunks(list(tx_to_genes), 50):
        res = _ensembl_post("/sequence/id", {"ids": batch}, params={"type": seq_type})
        for rec in res:
            for sym in tx_to_genes[rec["query"]]:
                lengths[sym] = len(rec["seq"])
        time.sleep(0.2)

    missing = sorted(set(genes) - set(lengths))
    if missing:
        print(f"No CDS length for {len(missing)} gene(s): {missing}")
    return lengths

## 4. End-to-end: `analyze(study_id)`

Combines the steps, computes the normalized metrics, draws the plots, and returns the per-gene DataFrame.

* **`freq_per_kb`** = `freq_pct / cds_kb` — headline length-normalized frequency.
* **`mut_per_sample_per_kb`** = `(total_mutations / n) / cds_kb` — mutation-density view.

In [ ]:
def analyze(study_id, top_n=TOP_N, plot=True):
    freq = get_mutation_frequencies(study_id, top_n=top_n)
    n_profiled = freq.attrs["n_profiled"]

    top = freq.copy()
    top["cds_bp"] = pd.Series(get_cds_lengths(top.index.tolist()))
    top = top.dropna(subset=["cds_bp"]).copy()
    top["cds_kb"] = top["cds_bp"] / 1000.0
    top["freq_per_kb"] = top["freq_pct"] / top["cds_kb"]
    top["mut_per_sample_per_kb"] = (top["total_mutations"] / n_profiled) / top["cds_kb"]
    top.attrs.update(freq.attrs)

    if plot:
        _plot_compare(top, study_id, n_profiled)
        _plot_scatter(top, study_id)
    return top


def _plot_compare(top, study_id, n, k=15):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    top.sort_values("freq_pct", ascending=False).head(k)["freq_pct"].plot(
        kind="bar", color="#4C72B0", ax=axes[0])
    axes[0].set_title(f"UNNORMALIZED\n{study_id} (n={n})")
    axes[0].set_ylabel("% samples mutated"); axes[0].set_xlabel("")
    top.sort_values("freq_per_kb", ascending=False).head(k)["freq_per_kb"].plot(
        kind="bar", color="#C44E52", ax=axes[1])
    axes[1].set_title(f"LENGTH-NORMALIZED (per kb CDS)\n{study_id} (n={n})")
    axes[1].set_ylabel("% samples mutated per kb CDS"); axes[1].set_xlabel("")
    for ax in axes:
        ax.tick_params(axis="x", rotation=300)
    sns.despine(); plt.tight_layout(); plt.show()


def _plot_scatter(top, study_id, label_n=8):
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(top["cds_kb"], top["freq_pct"], s=25, color="#55A868", alpha=0.7)
    slope = top["freq_pct"].sum() / top["cds_kb"].sum()
    xs = [0, top["cds_kb"].max()]
    ax.plot(xs, [slope * x for x in xs], "--", color="gray", label="expected ∝ CDS length")
    resid = top["freq_pct"] - slope * top["cds_kb"]
    for g in resid.sort_values(ascending=False).head(label_n).index:
        ax.annotate(g, (top.loc[g, "cds_kb"], top.loc[g, "freq_pct"]),
                    fontsize=9, xytext=(4, 3), textcoords="offset points")
    ax.set_xlabel("Canonical CDS length (kb)"); ax.set_ylabel("% samples mutated")
    ax.set_title(f"Mutation frequency vs gene length — {study_id}")
    ax.legend(); sns.despine(); plt.tight_layout(); plt.show()

### Run it on the selected study

In [ ]:
result = analyze(STUDY_ID, TOP_N)
result.head(15)

### Who moved most after normalization?

In [ ]:
def rank_changes(top):
    c = top.assign(
        rank_raw=top["freq_pct"].rank(ascending=False, method="min").astype(int),
        rank_norm=top["freq_per_kb"].rank(ascending=False, method="min").astype(int))
    c["rank_change"] = c["rank_raw"] - c["rank_norm"]   # +ve = climbed after normalizing
    return c[["freq_pct", "cds_bp", "freq_per_kb", "rank_raw", "rank_norm", "rank_change"]]

rc = rank_changes(result)
print("Biggest CLIMBERS (compact, recurrently mutated):")
display(rc.sort_values("rank_change", ascending=False).head(10).round(2))
print("\nBiggest FALLERS (long genes, likely passengers):")
display(rc.sort_values("rank_change").head(10).round(2))

## (Optional) Compare two studies side by side

Because the pipeline is a function, you can run it on several cohorts and compare. A targeted panel (MSK-IMPACT) vs a whole-exome cohort (TCGA) makes the length bias obvious — the big passenger genes (TTN, MUC16 …) only exist in the exome data.

In [ ]:
# Uncomment to compare. (Each call hits the APIs, so it takes a little while.)
# a = analyze("luad_mskimpact_2021", TOP_N, plot=False)   # MSK-IMPACT (targeted panel)
# b = analyze("luad_tcga_pub",       TOP_N, plot=False)   # TCGA LUAD (whole exome)
#
# merged = (a[["freq_pct", "freq_per_kb"]].add_suffix("_msk")
#           .join(b[["freq_pct", "freq_per_kb"]].add_suffix("_tcga"), how="outer"))
# merged.sort_values("freq_per_kb_tcga", ascending=False).head(20).round(2)

## Notes & caveats

* **Any study works** — use `find_studies("...")` to get an id, set `STUDY_ID`, rerun. `analyze()` auto-discovers each study's mutation profile and profiled-sample denominator.
* **Panel vs exome.** Targeted panels (MSK-IMPACT) omit giant passenger genes, so the length bias is milder than in whole-exome cohorts (TCGA). Compare them with the cell above.
* **CDS choice.** We use the Ensembl *canonical* transcript CDS. Pass `seq_type="protein"` to `get_cds_lengths` to normalize by amino acids (≈ CDS/3) instead.
* **Model.** `freq_per_kb` is a first-order, length-only correction — not a full background-mutation model (cf. MutSigCV, which also models per-base rate and sequence context).
* **`TOP_N`** controls how many genes flow through length lookup and plotting.